# Advanced document indexing

## Splitting and ingesting the content of a single URL (on Cornwall)

### Preparing the Chroma DB collections

In [1]:
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [2]:
ollama_embeddings = OllamaEmbeddings(model="bge-m3")

cornwall_granular_collection = Chroma(  # A
    collection_name="cornwall_granular",
    embedding_function=ollama_embeddings,
)

cornwall_granular_collection.reset_collection()  # B

# A Create a Chroma collection using local Ollama embeddings.
# B Reset the collection in case it already exists.

In [3]:
cornwall_coarse_collection = Chroma( # A 
    collection_name="cornwall_coarse",
    embedding_function=ollama_embeddings
)

cornwall_coarse_collection.reset_collection() # B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

### Loading the HTML content with the AsyncHtmlLoader

In [4]:
from langchain_community.document_loaders import AsyncHtmlLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()
len(docs)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.76it/s]


1

### Splitting into granular chunks with the HTMLSectionSplitter

In [6]:
from langchain_text_splitters import HTMLSectionSplitter

In [7]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks) 

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

granular_chunks = split_docs_into_granular_chunks(docs)

# Ingesting granular chunks
cornwall_granular_collection.add_documents(documents=granular_chunks)

# Searching granular chunks
results = cornwall_granular_collection.similarity_search(query="Events or festivals in Cornwall", k=3)
for doc in results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance 

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter

In [8]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=300)

def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A 
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

coarse_chunks = split_docs_into_coarse_chunks(docs)

# Ingesting coarse chunks
cornwall_coarse_collection.add_documents(documents=coarse_chunks)

# Searching coarse chunks
results = cornwall_coarse_collection.similarity_search(query="Events or festivals in Cornwall", k=3)
for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corni

## Splitting and ingesting the content of various URLs (across UK destinations)

In [10]:
# Preparing the Chroma DB collections
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=ollama_embeddings
)

uk_granular_collection.reset_collection() #B
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=ollama_embeddings
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter

In [11]:
# Reduce this list if you want to save on processing cost
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #C
    docs =  html_loader.load() #D
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists 
#C Loader for one destination
#D Documents of one destination

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.87it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.16it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.14it/s]


{'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.20it/s]


{'source': 'https://en.wikivoyage.org/wiki/West_Cornwall', 'title': 'West Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


In [12]:
# Searching
granular_results = uk_granular_collection.similarity_search(query="Events or festivals in East Sussex", k=4)
for doc in granular_results:
    print(doc)

page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance of associated rituals. Some towns have a street-parade dur

In [13]:
coarse_results = uk_coarse_collection.similarity_search(query="Events or festivals in East Sussex", k=4)
for doc in coarse_results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corni

In [14]:
granular_results = uk_granular_collection.similarity_search(query="Beaches in Cornwall", k=4)
for doc in granular_results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='North Cornwall' metadata={'Header 1': 'North Cornwall'}
page_content='West Cornwall' metadata={'Header 1': 'West Cornwall'}
page_content='South Cornwall' metadata={'Header 1': 'South Cornwall'}


In [15]:
coarse_results = uk_coarse_collection.similarity_search(query="Beaches in Cornwall", k=4)
for doc in coarse_results:
    print(doc)

page_content='**South Cornwall** is in Cornwall. It includes much of the stunning Cornish
coast along the English Channel of the Atlantic Ocean.

## Towns and villages

[edit]

Map of South Cornwall

  * 50.26-5.0511 Truro — Cornwall's main centre hosts the Royal Cornwall Museum
  * 50.3311-4.20212 Cawsand — overlooks Plymouth Sound; Cawsand is within Mount Edgcumbe Country Park
  * 50.15-5.073 Falmouth — famous for its beaches, it is home to the world's third largest natural harbour
  * 50.334-4.6334 Fowey — the Fowey Regatta in mid-August attracts many yachts and sailing boats
  * 50.354-4.4545 Looe — a summer resort place with a monkey sanctuary, and an active fishing village
  * 50.408-4.2126 Saltash — "Gateway to Cornwall", a small town on the Cornwall side of the Tamar crossings
  * 50.338-4.7957 St Austell — largest town in the county and home to the Eden Project, the world's largest greenhouse
  * 50.3314-4.75788 Charlestown — seaside town used as filming location for the TV sh

## Embedding strategy

### Embedding child chunks with ParentDocumentRetriever

In [16]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [18]:
# Setting up the Parent Document retriever

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=ollama_embeddings,
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

In [19]:
# Ingesting the content into doc and vector store

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination 
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.85it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.16it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.23it/s]


Ingesting https://en.wikivoyage.org/wiki/South_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.97it/s]


Ingesting https://en.wikivoyage.org/wiki/West_Cornwall


In [20]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['2f60d052-858e-4a04-ba22-736ce46f5da3',
 'b91bc7f2-8000-41e1-ae00-04a9de7abc91',
 '53b56f78-54c0-441e-b122-f425461e6198',
 '0e514846-5d03-414a-b4fd-6b98df19548c',
 '8c1847b0-a0a1-49cf-8505-4bf6a496af84',
 '07897a71-995c-4f5c-b721-5ef3d8b5a596',
 'e564b18b-cae6-4320-8f22-1ec8794c4814',
 'a741ac8c-7230-4758-8841-3ec1f5c82e53',
 '454dcea1-f487-462f-b4b4-6120bf55369e',
 '09fbe0c1-f970-4301-b2fa-e46a45f79442',
 '29b76e27-edf5-4b6c-9aec-8289a059c705',
 '94609177-1d77-489e-901d-3fbf47a71101',
 'd801da87-7ed1-4bbb-976d-56d10c11ba1b',
 '8d62c038-ed27-4510-b5e3-b6ba514378a4',
 '988e0362-04a5-4f86-bbe4-2d52d261eba3',
 'baaebe4f-f944-47e0-aff0-882da7639e39',
 '3ab60f2a-4254-4d49-bd3a-b581e9f3c3d6',
 '44a06a79-93cf-4826-83d6-d906de046f64',
 '807ead70-1445-4ac7-ae88-48e9b99e235a',
 '9c4e14bd-40f5-4683-a34d-cd2266a2f04b',
 'b17e822b-af65-4b31-b1a0-75c2607ff35d',
 'a8e2f31f-6e75-4a64-9f23-502a14908cb5',
 '5a080e34-30ab-4c99-a4f8-e71ca24d2965',
 '85b1322d-003c-4b3c-89e5-292e83fc410e',
 'fe603075-5b7d-

In [21]:
# Performing a search on granular information

retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")
len(retrieved_docs)

4

In [22]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="**Go Cornwall Bus** buses operate between:\n\n  * **10** \\- Plymouth to Saltash, Looe and Polperro\n  * **11** \\- Plymouth to Saltash, Liskeard, Bodmin, Wadebridge and Padstow\n\n**Stagecoach** buses operate between Barnstaple, Holsworthy, Launceston and\nTavistock, across the Cornwall and Devon border (**85**).\n\n## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies (except certain town buses in St Ives and Fowey). The\n**Cornwall All Day ticket** allows unlimited travel for a calendar day. As of\n2025, day passes are £8 for adults and £5 for under-19s and £3 singles,\nregardless of age. Payment is by cash or contactless. Real time information\nand timetables can now be found through Transport for Cornwall (most reliable\nfor real t

In [23]:
# Comparing with direct semantic search on child chunks

child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")
print(len(child_docs_only))
child_docs_only[0]

4


Document(id='1477897e-6be8-42dd-b9fd-561e1e46ebfe', metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en', 'doc_id': 'a741ac8c-7230-4758-8841-3ec1f5c82e53'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\nThere is also a new Pay as you Go smartcard system for Cornwall, run by GWR\nbut valid on all trains in Cornwall.\n\n### By ferry/boat\n\n[edit]')

### Embedding child chunks with MultiVectorRetriever

In [24]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

In [25]:
ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [26]:
# Setting up the Multi vector retriever

from langchain_community.embeddings import OpenAIEmbeddings


parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=ollama_embeddings,
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

In [27]:
import time
from ollama import ResponseError


def add_documents_in_batches(
    vectorstore,
    documents,
    batch_size=8,
    max_retries=3,
):
    """Embed and store documents in small, retryable batches."""

    for start in range(0, len(documents), batch_size):
        batch = documents[start:start + batch_size]

        for attempt in range(max_retries):
            try:
                vectorstore.add_documents(batch)
                break

            except ResponseError:
                if attempt == max_retries - 1:
                    raise

                delay = 2 ** attempt
                print(
                    f"Embedding batch failed; retrying in {delay} second(s)..."
                )
                time.sleep(delay)


# Ingesting the content into document and vector stores

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)  # A
    html_docs = html_loader.load()                  # B
    text_docs = html2text_transformer.transform_documents(
        html_docs
    )                                               # C

    coarse_chunks = parent_splitter.split_documents(
        text_docs
    )                                               # D

    coarse_chunks_ids = [
        str(uuid.uuid4()) for _ in coarse_chunks
    ]

    all_granular_chunks = []

    for i, coarse_chunk in enumerate(coarse_chunks):  # E
        coarse_chunk_id = coarse_chunks_ids[i]

        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]
        )                                             # F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id  # G

        all_granular_chunks.extend(granular_chunks)

    print(
        f"Ingesting {destination_url}: "
        f"{len(coarse_chunks)} parent chunks, "
        f"{len(all_granular_chunks)} child chunks"
    )

    add_documents_in_batches(
        vectorstore=multi_vector_retriever.vectorstore,
        documents=all_granular_chunks,
        batch_size=8,
    )                                                 # H

    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))
    )                                                 # I

# A Load one destination page.
# B Retrieve its HTML document.
# C Transform HTML into clean text.
# D Create coarse parent chunks.
# E Iterate over parent chunks.
# F Create granular child chunks.
# G Link every child chunk to its parent.
# H Embed and store child chunks in small batches.
# I Store parent chunks after all corresponding children succeed.

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.86it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall: 15 parent chunks, 133 child chunks


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.22it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall: 6 parent chunks, 45 child chunks


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.26it/s]


Ingesting https://en.wikivoyage.org/wiki/South_Cornwall: 4 parent chunks, 32 child chunks


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.17it/s]


Ingesting https://en.wikivoyage.org/wiki/West_Cornwall: 6 parent chunks, 42 child chunks


In [28]:
# Performing a search on granular information

retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
len(retrieved_docs)

4

In [29]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="**Go Cornwall Bus** buses operate between:\n\n  * **10** \\- Plymouth to Saltash, Looe and Polperro\n  * **11** \\- Plymouth to Saltash, Liskeard, Bodmin, Wadebridge and Padstow\n\n**Stagecoach** buses operate between Barnstaple, Holsworthy, Launceston and\nTavistock, across the Cornwall and Devon border (**85**).\n\n## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies (except certain town buses in St Ives and Fowey). The\n**Cornwall All Day ticket** allows unlimited travel for a calendar day. As of\n2025, day passes are £8 for adults and £5 for under-19s and £3 singles,\nregardless of age. Payment is by cash or contactless. Real time information\nand timetables can now be found through Transport for Cornwall (most reliable\nfor real t

Note: **Same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks.** [See](naive-rag-issues-and-solutions.md)

In [30]:
# Comparing with direct semantic search on child chunks

child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")
len(child_docs_only)

4

In [31]:
child_docs_only[0]

Document(id='02a63da4-3dbe-423c-9771-75750d635eca', metadata={'language': 'en', 'title': 'Cornwall – Travel guide at Wikivoyage', 'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'doc_id': '54b599bf-4d56-475e-b621-fa8857ba1646'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\nThere is also a new Pay as you Go smartcard system for Cornwall, run by GWR\nbut valid on all trains in Cornwall.\n\n### By ferry/boat\n\n[edit]')

### Embedding summaries with MultiVectorRetriever

In [32]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

In [33]:
ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [34]:
# Setting up the Multi vector retriever (similar to when embedding child chunks)

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

summaries_collection = Chroma( #B
    collection_name="uk_summaries",
    embedding_function=ollama_embeddings,
)

summaries_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to summary vectors

In [35]:
# Setting up the summarization chain

from langchain_community.chat_models import ChatOpenAI

llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    keep_alive=1800, # keeps the model loaded for 30 minutes.
)

summarization_chain = (
    {"document": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template(
        """Summarize the following travel document in at most 120 words.
            Preserve important place names, attractions, activities, and practical details.

            Document:
            {document}"""
        ) #B
    | llm
    | StrOutputParser())

#A Grab the text content from the document
#B Instantiate a prompt asking to generate summary of the provided text
#C Send the LLM the instantiated prompt 
#D Extract the summary text from the response

In [36]:
import time
import httpx

from ollama import ResponseError


TRANSIENT_ERRORS = (
    ResponseError,
    httpx.HTTPError,
    ConnectionError,
)


def summarize_with_retry(
    document,
    max_retries=3,
):
    """Generate one summary with exponential-backoff retries."""

    for attempt in range(max_retries):
        try:
            return summarization_chain.invoke(document)

        except TRANSIENT_ERRORS:
            if attempt == max_retries - 1:
                raise

            delay = 2 ** attempt
            print(
                f"Summary request failed; "
                f"retrying in {delay} second(s)..."
            )
            time.sleep(delay)


def add_summary_documents_in_batches(
    vectorstore,
    documents,
    document_ids,
    batch_size=8,
    max_retries=3,
):
    """Embed and store summaries in small retryable batches."""

    total = len(documents)

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)

        batch_documents = documents[start:end]
        batch_ids = document_ids[start:end]

        for attempt in range(max_retries):
            try:
                vectorstore.add_documents(
                    documents=batch_documents,
                    ids=batch_ids,
                )

                print(
                    f"Embedded summaries {start + 1}-{end}/{total}"
                )
                break

            except TRANSIENT_ERRORS:
                if attempt == max_retries - 1:
                    raise

                delay = 2 ** attempt
                print(
                    f"Embedding summaries {start + 1}-{end} failed; "
                    f"retrying in {delay} second(s)..."
                )
                time.sleep(delay)

In [37]:
# Ingesting coarse chunks and related summaries into doc and vector store

for destination_url in uk_destination_urls:
    print(f"\nLoading {destination_url}")

    html_loader = AsyncHtmlLoader(destination_url)  # A
    html_docs = html_loader.load()                  # B

    text_docs = html2text_transformer.transform_documents(
        html_docs
    )                                               # C

    coarse_chunks = parent_splitter.split_documents(
        text_docs
    )                                               # D

    # Stable IDs prevent duplicates if ingestion is retried.
    coarse_chunks_ids = [
        str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"{destination_url}#chunk={chunk_index}",
            )
        )
        for chunk_index in range(len(coarse_chunks))
    ]

    all_summaries = []
    total_chunks = len(coarse_chunks)

    for chunk_index, coarse_chunk in enumerate(coarse_chunks):
        print(
            f"Summarizing chunk "
            f"{chunk_index + 1}/{total_chunks}"
        )

        summary_text = summarize_with_retry(
            coarse_chunk
        )                                           # E

        summary_doc = Document(
            page_content=summary_text,
            metadata={
                doc_key: coarse_chunks_ids[chunk_index]
            },
        )

        all_summaries.append(summary_doc)           # F

        print(
            f"Completed chunk "
            f"{chunk_index + 1}/{total_chunks}"
        )

    print(
        f"Embedding {len(all_summaries)} summaries "
        f"for {destination_url}"
    )

    add_summary_documents_in_batches(
        vectorstore=multi_vector_retriever.vectorstore,
        documents=all_summaries,
        document_ids=coarse_chunks_ids,
        batch_size=8,
    )                                               # G

    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))
    )                                               # H

    print(f"Finished ingesting {destination_url}")

# A Load one destination page.
# B Retrieve its HTML.
# C Transform HTML into clean text.
# D Create coarse parent chunks.
# E Generate each summary with retries.
# F Link each summary to its parent chunk.
# G Embed summaries in small retryable batches.
# H Store the original parent chunks.


Loading https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.81it/s]


Summarizing chunk 1/15
Completed chunk 1/15
Summarizing chunk 2/15
Completed chunk 2/15
Summarizing chunk 3/15
Completed chunk 3/15
Summarizing chunk 4/15
Completed chunk 4/15
Summarizing chunk 5/15
Completed chunk 5/15
Summarizing chunk 6/15
Completed chunk 6/15
Summarizing chunk 7/15
Completed chunk 7/15
Summarizing chunk 8/15
Completed chunk 8/15
Summarizing chunk 9/15
Completed chunk 9/15
Summarizing chunk 10/15
Completed chunk 10/15
Summarizing chunk 11/15
Completed chunk 11/15
Summarizing chunk 12/15
Completed chunk 12/15
Summarizing chunk 13/15
Completed chunk 13/15
Summarizing chunk 14/15
Completed chunk 14/15
Summarizing chunk 15/15
Completed chunk 15/15
Embedding 15 summaries for https://en.wikivoyage.org/wiki/Cornwall
Embedded summaries 1-8/15
Embedded summaries 9-15/15
Finished ingesting https://en.wikivoyage.org/wiki/Cornwall

Loading https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.84it/s]


Summarizing chunk 1/6
Completed chunk 1/6
Summarizing chunk 2/6
Completed chunk 2/6
Summarizing chunk 3/6
Completed chunk 3/6
Summarizing chunk 4/6
Completed chunk 4/6
Summarizing chunk 5/6
Completed chunk 5/6
Summarizing chunk 6/6
Completed chunk 6/6
Embedding 6 summaries for https://en.wikivoyage.org/wiki/North_Cornwall
Embedded summaries 1-6/6
Finished ingesting https://en.wikivoyage.org/wiki/North_Cornwall

Loading https://en.wikivoyage.org/wiki/South_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.24it/s]


Summarizing chunk 1/4
Completed chunk 1/4
Summarizing chunk 2/4
Completed chunk 2/4
Summarizing chunk 3/4
Completed chunk 3/4
Summarizing chunk 4/4
Completed chunk 4/4
Embedding 4 summaries for https://en.wikivoyage.org/wiki/South_Cornwall
Embedded summaries 1-4/4
Finished ingesting https://en.wikivoyage.org/wiki/South_Cornwall

Loading https://en.wikivoyage.org/wiki/West_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.27it/s]


Summarizing chunk 1/6
Completed chunk 1/6
Summarizing chunk 2/6
Completed chunk 2/6
Summarizing chunk 3/6
Completed chunk 3/6
Summarizing chunk 4/6
Completed chunk 4/6
Summarizing chunk 5/6
Completed chunk 5/6
Summarizing chunk 6/6
Completed chunk 6/6
Embedding 6 summaries for https://en.wikivoyage.org/wiki/West_Cornwall
Embedded summaries 1-6/6
Finished ingesting https://en.wikivoyage.org/wiki/West_Cornwall


Everything is working correctly. It has taken *2 minutes 17 seconds* to run on my laptop *(I have decreased the number of destination urls and limited local LLM's settings to use less tokens)*.

Successfully completed:

- Cornwall: 15 summaries, embedded in two batches (`1–8`, `9–15`)
- North Cornwall: 6 summaries, one batch
- South Cornwall: 4 summaries, one batch
- West Cornwall: 6 summaries, one batch

No retry messages appeared, so all Ollama generation and BGE-M3 embedding requests succeeded on their first attempt.

For each destination, `Finished ingesting ...` confirms:

1. The page loaded.
2. All coarse chunks were summarized.
3. Every summary was embedded into Chroma.
4. The corresponding original coarse chunks were stored in the document store.

In [38]:
# Performing a search on granular information

retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")
len(retrieved_docs)

4

In [39]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies. The **Cornwall All Day ticket** allows unlimited\ntravel for a calendar day. As of 2023, fares are £5 for adults and £4 for\nunder-19s. Payment is by cash or contactless. The two main bus companies are:\n\n  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).\n  * **Kernow** (part of First Bus) covers western and central Cornwall.\n\nBuses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\ns

In [40]:
# Comparing with direct semantic search on summaries

summary_docs_only =  summaries_collection.similarity_search("Cornwall Travel")
len(summary_docs_only)

4

In [41]:
summary_docs_only

[Document(id='be26e114-df79-5bf3-b1b4-7c233651a4b3', metadata={'doc_id': 'be26e114-df79-5bf3-b1b4-7c233651a4b3'}, page_content="Travelers can navigate Cornwall using **Go Cornwall Bus** or **Kernow**, utilizing the **Cornwall All Day ticket** (£5 adults). Trains operated by **CrossCountry** and **Great Western Railway** are available, with the **Cornwall Ranger** ticket offering unlimited daily travel for £14.\n\nKey attractions include the **Eden Project**, **Lost Gardens of Heligan**, and the **National Maritime Museum Falmouth**. Notable National Trust sites include **Antony House**, **Cotehele - St Dominick**, **Trelissick**, and **Glendurgan**. For activities, hikers can explore the scenic **South West Coast Path**, especially near Penwith and the Lizard. Visitors can also celebrate **St Piran's Day** on March 5th with regional festivities."),
 Document(id='4aa28ae8-ab35-511f-b6ad-54e19fc4e0d4', metadata={'doc_id': '4aa28ae8-ab35-511f-b6ad-54e19fc4e0d4'}, page_content='Cornwall is

### Embedding hypothetical questions with MultiVectorRetriever

In [1]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [3]:
# Setting up the Multi vector retriever (same as when embedding summaries)

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

hypothetical_questions_collection = Chroma( #B
    collection_name="uk_hypothetical_questions",
    embedding_function=ollama_embeddings,
)

hypothetical_questions_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child hypothetical questions
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child hypothetical questions

In [4]:
# Setting up the chain to generate hypothetical questions

class HypotheticalQuestions(BaseModel):
    """Hypothetical questions answerable from a document."""

    questions: List[str] = Field(
        ...,
        min_length=4,
        max_length=4,
        description="Exactly four questions answerable from the document",
    )


question_generation_llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192,
    num_predict=192,
    temperature=0,
    reasoning=False,
    keep_alive=1800,
)

llm_with_structured_output = (
    question_generation_llm.with_structured_output(
        HypotheticalQuestions
    )
)


hypothetical_questions_chain = (
    {"document_text": lambda document: document.page_content} #A
    | ChatPromptTemplate.from_template( #B
        """Generate exactly four distinct hypothetical questions that can
            be answered using only the following document.

            Document:
            {document_text}"""
    )
    | llm_with_structured_output #C
    | (lambda result: result.questions) #D
)

#A Grab the text content from the document
#B Instantiate a prompt asking to generate 4 hypothetical questions on the provided text
#C Invoke the LLM configured to return an object containing the questions as a typed list of strings
#D Grab the list of questions from the response

In [5]:
import time
import httpx

from ollama import ResponseError
from pydantic import ValidationError
from langchain_core.exceptions import OutputParserException


QUESTION_GENERATION_ERRORS = (
    ResponseError,
    httpx.HTTPError,
    ConnectionError,
    ValidationError,
    OutputParserException,
)

EMBEDDING_ERRORS = (
    ResponseError,
    httpx.HTTPError,
    ConnectionError,
)


def generate_questions_with_retry(
    document,
    max_retries=3,
):
    """Generate four hypothetical questions with retries."""

    for attempt in range(max_retries):
        try:
            return hypothetical_questions_chain.invoke(document)

        except QUESTION_GENERATION_ERRORS as error:
            if attempt == max_retries - 1:
                raise

            delay = 2 ** attempt
            print(
                f"Question generation failed: {error}. "
                f"Retrying in {delay} second(s)..."
            )
            time.sleep(delay)


def add_question_documents_in_batches(
    vectorstore,
    documents,
    document_ids,
    batch_size=8,
    max_retries=3,
):
    """Embed and upsert question documents in retryable batches."""

    total = len(documents)

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)

        batch_documents = documents[start:end]
        batch_ids = document_ids[start:end]

        for attempt in range(max_retries):
            try:
                vectorstore.add_documents(
                    documents=batch_documents,
                    ids=batch_ids,
                )

                print(
                    f"Embedded hypothetical questions "
                    f"{start + 1}-{end}/{total}"
                )
                break

            except EMBEDDING_ERRORS as error:
                if attempt == max_retries - 1:
                    raise

                delay = 2 ** attempt
                print(
                    f"Embedding batch {start + 1}-{end} failed: "
                    f"{error}. Retrying in {delay} second(s)..."
                )
                time.sleep(delay)

In [9]:
# Ingesting the parent coarse chunks and their hypothetical questions into doc and vector store

from langchain_community.document_transformers import Html2TextTransformer

html2text_transformer = Html2TextTransformer()

# Reduce this list if you want to save on processing cost
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

for destination_url in uk_destination_urls:
    print(f"\nLoading {destination_url}")

    html_loader = AsyncHtmlLoader(destination_url)  # A
    html_docs = html_loader.load()                  # B

    text_docs = html2text_transformer.transform_documents(
        html_docs
    )                                               # C

    coarse_chunks = parent_splitter.split_documents(
        text_docs
    )                                               # D

    total_chunks = len(coarse_chunks)

    # Deterministic parent IDs make retries idempotent.
    coarse_chunks_ids = [
        str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"{destination_url}#parent={chunk_index}",
            )
        )
        for chunk_index in range(total_chunks)
    ]

    all_question_documents = []
    all_question_ids = []

    for chunk_index, coarse_chunk in enumerate(coarse_chunks):
        coarse_chunk_id = coarse_chunks_ids[chunk_index]

        print(
            f"Generating questions for parent chunk "
            f"{chunk_index + 1}/{total_chunks}"
        )

        hypothetical_questions = generate_questions_with_retry(
            coarse_chunk
        )                                               # E

        for question_index, question in enumerate(
            hypothetical_questions
        ):
            # Deterministic vector ID prevents duplicates on retries.
            question_id = str(
                uuid.uuid5(
                    uuid.NAMESPACE_URL,
                    f"{coarse_chunk_id}#question={question_index}",
                )
            )

            question_document = Document(
                page_content=question,
                metadata={
                    doc_key: coarse_chunk_id,
                    "source": destination_url,
                    "question_index": question_index,
                },
            )

            all_question_documents.append(question_document)
            all_question_ids.append(question_id)       # F

        print(
            f"Completed parent chunk "
            f"{chunk_index + 1}/{total_chunks}: "
            f"{len(hypothetical_questions)} questions"
        )

    print(
        f"Embedding {len(all_question_documents)} hypothetical "
        f"questions for {destination_url}"
    )

    add_question_documents_in_batches(
        vectorstore=multi_vector_retriever.vectorstore,
        documents=all_question_documents,
        document_ids=all_question_ids,
        batch_size=8,
    )                                                   # G

    # Parents are stored only after all question vectors succeed.
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))
    )                                                   # H

    print(
        f"Finished ingesting {destination_url}: "
        f"{total_chunks} parents and "
        f"{len(all_question_documents)} questions"
    )

# A Load one destination page.
# B Retrieve its HTML.
# C Transform HTML into clean text.
# D Create coarse parent chunks.
# E Generate four hypothetical questions with retries.
# F Link every question to its parent and assign a stable vector ID.
# G Embed questions in small retryable batches.
# H Store parents after all corresponding question vectors succeed.


Loading https://en.wikivoyage.org/wiki/Cornwall


Fetching pages:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.81it/s]


Generating questions for parent chunk 1/15
Completed parent chunk 1/15: 4 questions
Generating questions for parent chunk 2/15
Completed parent chunk 2/15: 4 questions
Generating questions for parent chunk 3/15
Completed parent chunk 3/15: 4 questions
Generating questions for parent chunk 4/15
Completed parent chunk 4/15: 4 questions
Generating questions for parent chunk 5/15
Completed parent chunk 5/15: 4 questions
Generating questions for parent chunk 6/15
Completed parent chunk 6/15: 4 questions
Generating questions for parent chunk 7/15
Completed parent chunk 7/15: 4 questions
Generating questions for parent chunk 8/15
Completed parent chunk 8/15: 4 questions
Generating questions for parent chunk 9/15
Completed parent chunk 9/15: 4 questions
Generating questions for parent chunk 10/15
Completed parent chunk 10/15: 4 questions
Generating questions for parent chunk 11/15
Completed parent chunk 11/15: 4 questions
Generating questions for parent chunk 12/15
Completed parent chunk 12/15

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.06it/s]


Generating questions for parent chunk 1/6
Completed parent chunk 1/6: 4 questions
Generating questions for parent chunk 2/6
Completed parent chunk 2/6: 4 questions
Generating questions for parent chunk 3/6
Completed parent chunk 3/6: 4 questions
Generating questions for parent chunk 4/6
Completed parent chunk 4/6: 4 questions
Generating questions for parent chunk 5/6
Completed parent chunk 5/6: 4 questions
Generating questions for parent chunk 6/6
Completed parent chunk 6/6: 4 questions
Embedding 24 hypothetical questions for https://en.wikivoyage.org/wiki/North_Cornwall
Embedded hypothetical questions 1-8/24
Embedded hypothetical questions 9-16/24
Embedded hypothetical questions 17-24/24
Finished ingesting https://en.wikivoyage.org/wiki/North_Cornwall: 6 parents and 24 questions

Loading https://en.wikivoyage.org/wiki/South_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.51it/s]


Generating questions for parent chunk 1/4
Completed parent chunk 1/4: 4 questions
Generating questions for parent chunk 2/4
Completed parent chunk 2/4: 4 questions
Generating questions for parent chunk 3/4
Completed parent chunk 3/4: 4 questions
Generating questions for parent chunk 4/4
Completed parent chunk 4/4: 4 questions
Embedding 16 hypothetical questions for https://en.wikivoyage.org/wiki/South_Cornwall
Embedded hypothetical questions 1-8/16
Embedded hypothetical questions 9-16/16
Finished ingesting https://en.wikivoyage.org/wiki/South_Cornwall: 4 parents and 16 questions

Loading https://en.wikivoyage.org/wiki/West_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.41it/s]


Generating questions for parent chunk 1/6
Completed parent chunk 1/6: 4 questions
Generating questions for parent chunk 2/6
Completed parent chunk 2/6: 4 questions
Generating questions for parent chunk 3/6
Completed parent chunk 3/6: 4 questions
Generating questions for parent chunk 4/6
Completed parent chunk 4/6: 4 questions
Generating questions for parent chunk 5/6
Completed parent chunk 5/6: 4 questions
Generating questions for parent chunk 6/6
Completed parent chunk 6/6: 4 questions
Embedding 24 hypothetical questions for https://en.wikivoyage.org/wiki/West_Cornwall
Embedded hypothetical questions 1-8/24
Embedded hypothetical questions 9-16/24
Embedded hypothetical questions 17-24/24
Finished ingesting https://en.wikivoyage.org/wiki/West_Cornwall: 6 parents and 24 questions


The hypothetical-question ingestion completed successfully for all four destinations:

| Destination | Parent chunks | Generated questions | Embedding batches |
|---|---:|---:|---:|
| Cornwall | 15 | 60 | 8 |
| North Cornwall | 6 | 24 | 3 |
| South Cornwall | 4 | 16 | 2 |
| West Cornwall | 6 | 24 | 3 |
| **Total** | **31** | **124** | **16** |

The output confirms that:

- Exactly four questions were generated for every parent chunk.
- Structured output validation succeeded.
- All 124 questions were embedded in batches of at most eight.
- No retry messages appeared, so every generation and embedding request succeeded on its first attempt.
- Each question contains its parent `doc_id`.
- All 31 original parent chunks were stored after their associated question vectors succeeded.

In [12]:
# Performing a search on granular information

retrieved_docs = multi_vector_retriever.invoke(
    "How long does it take to travel from London to Penzance by train?")

print(len(retrieved_docs))
print(retrieved_docs)

4
[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/West_Cornwall', 'title': 'West Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content='## Towns and villages\n\n[edit]\n\nMap of West Cornwall\n\n  * 49.988-5.181 Cadgwith — a popular summer holiday destination along the South West Coast Path\n  * 50.183-5.4162 Hayle — the townscape of Hayle and its historic harbour are part of the Cornwall and West Devon Historic Mining Landscape World Heritage site\n  * 50.101563-5.2775093 Helston — a gateway to the Lizard Peninsula, famous for its Flora Day celebrations and Furry Dance\n  * 50.125-5.4764 Marazion — the tidal island of St Michael\'s Mount is half-a-mile offshore\n  * 50.101-5.5535 Newlyn — it has many charming cottages and narrow lanes to explore, and is home to a collection of modern art\n  * 50.119-5.5376 Penzance — pirate central, Penzance is a town long-associated with the arts\n  * 50.084-5.3157 Porthleven — it is one of Britain\'s best-known surfin

In [13]:
# Inspecting possible questions matching our question through semantic search

hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search(
    "How long does it take to travel from London to Penzance by train?")

print(len(hypothetical_question_docs_only))
print(hypothetical_question_docs_only)

4
[Document(id='c9617151-5f7d-5b0f-bbac-55f7c4b74612', metadata={'source': 'https://en.wikivoyage.org/wiki/West_Cornwall', 'doc_id': '55d0c965-3850-5e71-9242-d18fba9aec30', 'question_index': 2}, page_content='How long does it take to travel from London Paddington to Penzance by train on the main line?'), Document(id='74f3cf3e-d3f4-5795-95d7-818b2a082dce', metadata={'doc_id': 'cc2aba76-0e46-52fb-bea4-2a59eedc51ba', 'question_index': 3, 'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='What is the approximate road distance from Birmingham to Penzance?'), Document(id='a6965bd4-a953-5660-8e6c-515e460aca3d', metadata={'question_index': 0, 'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'doc_id': '8a622d27-b95f-5f80-ad06-aa5b6f02064f'}, page_content="What is the distance by road from Penzance to Land's End Airport?"), Document(id='7ab3f3f9-8353-5115-81dd-1ffc15bd4da6', metadata={'doc_id': '5177699c-9206-5209-be05-e6e33deb81f1', 'question_index': 2, 'source': 'https://e

The supported-query test is successful. The pipeline retrieved the correct evidence at rank 1.

Query:

```text
How long does it take to travel from London to Penzance by train?
```

Top hypothetical-question match:

```text
How long does it take to travel from London Paddington
to Penzance by train on the main line?
```

Its `doc_id` correctly resolved to the West Cornwall parent chunk containing:

```text
Regular trains run on the main line from London Paddington
... to Penzance, 5 hr 30 min
```

Therefore, the grounded answer is:

> The main-line journey from London Paddington to Penzance takes approximately 5 hours and 30 minutes. The document also reports eight daily trains and an overnight sleeper running Sunday–Friday nights.

The complete successful path is:

```text
User query
  → nearly identical hypothetical question
  → parent doc_id
  → West Cornwall parent chunk
  → explicit “5 hr 30 min” answer
```

### Remaining results

The other three hypothetical-question matches were weaker:

1. Birmingham-to-Penzance road distance
2. Penzance-to-Land’s End Airport road distance
3. London-to-Plymouth train duration

They were selected because they share concepts such as London, Penzance, travel, distance, train and duration. The fourth result remains reasonably related; the middle two are mainly lexical/semantic fallbacks.

This happens because `similarity_search()` and the underlying retriever return four results by default, even if only the first is highly relevant. It does not indicate a pipeline error.

For this particular query, retrieving one or two candidates would produce a cleaner context:

```python
multi_vector_retriever.search_kwargs = {"k": 2}
```

For diagnostics, inspect distances:

```python
question_matches = (
    hypothetical_questions_collection.similarity_search_with_score(
        "How long does it take to travel from London to Penzance by train?",
        k=4,
    )
)

for document, distance in question_matches:
    print(f"Distance: {distance:.4f}")
    print(f"Question: {document.page_content}")
    print(f"Parent ID: {document.metadata[doc_key]}")
    print()
```

With Chroma’s distance-based result, lower values indicate closer matches.

Final conclusion: hypothetical-question generation, BGE-M3 retrieval, `doc_id` linkage and parent-document retrieval are all functioning correctly. This test provides strong rank-1 evidence that hypothetical-question indexing successfully maps a natural user query to the parent chunk containing the answer. The visible `â€“` characters remain a separate text-encoding issue.

In [14]:
question_matches = (
    hypothetical_questions_collection.similarity_search_with_score(
        "How long does it take to travel from London to Penzance by train?",
        k=4,
    )
)

for document, distance in question_matches:
    print(f"Distance: {distance:.4f}")
    print(f"Question: {document.page_content}")
    print(f"Parent ID: {document.metadata[doc_key]}")
    print()

Distance: 0.1665
Question: How long does it take to travel from London Paddington to Penzance by train on the main line?
Parent ID: 55d0c965-3850-5e71-9242-d18fba9aec30

Distance: 0.4360
Question: What is the approximate road distance from Birmingham to Penzance?
Parent ID: cc2aba76-0e46-52fb-bea4-2a59eedc51ba

Distance: 0.5168
Question: What is the distance by road from Penzance to Land's End Airport?
Parent ID: 8a622d27-b95f-5f80-ad06-aa5b6f02064f

Distance: 0.5600
Question: How long does it take for a train from London to reach Plymouth?
Parent ID: 5177699c-9206-5209-be05-e6e33deb81f1



This distance output confirms that retrieval is working well.

| Rank | Distance | Assessment |
|---:|---:|---|
| 1 | **0.1665** | Excellent match; asks essentially the same question |
| 2 | 0.4360 | Partial match through “Penzance” and travel distance |
| 3 | 0.5168 | Weaker match through “Penzance” and distance |
| 4 | 0.5600 | Partial match through “London,” “train” and journey time |

The rank-1 result is clearly separated from rank 2 by `0.2695`. Its parent ID resolves to the West Cornwall chunk containing the explicit answer of approximately **5 hours 30 minutes**.

Because Chroma is using L2 distance here, smaller means closer. Distance is not a confidence percentage. Assuming Ollama’s unit-normalized embeddings, the approximate cosine similarities are:

| Distance | Approximate cosine similarity |
|---:|---:|
| 0.1665 | 0.917 |
| 0.4360 | 0.782 |
| 0.5168 | 0.742 |
| 0.5600 | 0.720 |

For unit vectors:

```text
cosine similarity ≈ 1 - (squared L2 distance / 2)
```

For this test, either `k=1` or an empirical raw-distance cutoff around `0.30` would isolate the directly relevant result:

```python
multi_vector_retriever.search_kwargs = {"k": 1}
```

A `0.30` threshold should not be treated as universal—it should be validated with several relevant and irrelevant queries.

Final conclusion: the hypothetical-question approach produced a near-paraphrase of the user query, ranked it decisively first, and correctly resolved it to the parent document containing the answer. This is strong evidence that the indexing and retrieval workflow operates as intended.

## Granular chunk expansion with MultiVectorRetriever

In [1]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import uuid

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [3]:
# Setting up the Multi vector retriever

granular_chunk_splitter = RecursiveCharacterTextSplitter( # defaults to `chunk_overlap=200`, tune it
    chunk_size=500,
) #A

granular_chunks_collection = Chroma( #B
    collection_name="uk_granular_chunks",
    embedding_function=ollama_embeddings,
)

granular_chunks_collection.reset_collection() #C

expanded_chunk_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)
#A Splitter to generate granular chunks from original documents (parsed from web pages);
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host expanded chunks
#E Retriever to link expanded chunks to child granular chunks

In [4]:
# Ingesting granular and expanded chunks into document and vector stores

import time
import httpx

from ollama import ResponseError
from langchain_community.document_transformers import Html2TextTransformer


EMBEDDING_ERRORS = (
    ResponseError,
    httpx.HTTPError,
    ConnectionError,
)


def add_granular_documents_in_batches(
    vectorstore,
    documents,
    document_ids,
    batch_size=8,
    max_retries=3,
):
    """Embed and upsert granular chunks in retryable batches."""

    total = len(documents)

    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch_documents = documents[start:end]
        batch_ids = document_ids[start:end]

        for attempt in range(max_retries):
            try:
                vectorstore.add_documents(
                    documents=batch_documents,
                    ids=batch_ids,
                )
                print(f"Embedded granular chunks {start + 1}-{end}/{total}")
                break

            except EMBEDDING_ERRORS as error:
                if attempt == max_retries - 1:
                    raise

                delay = 2 ** attempt
                print(
                    f"Embedding batch {start + 1}-{end} failed: {error}. "
                    f"Retrying in {delay} second(s)..."
                )
                time.sleep(delay)


html2text_transformer = Html2TextTransformer()

# Reduce this list if you want to save on processing cost
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f"{wikivoyage_root_url}/{d}" for d in uk_destinations]

for destination_url in uk_destination_urls:
    print(f"\nLoading {destination_url}")

    html_loader = AsyncHtmlLoader(destination_url)  # A
    html_docs = html_loader.load()                  # B
    text_docs = html2text_transformer.transform_documents(
        html_docs
    )                                               # C

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs
    )                                               # D

    total_chunks = len(granular_chunks)
    granular_chunk_ids = []
    expanded_chunk_store_items = []

    for chunk_index, granular_chunk in enumerate(granular_chunks):  # E
        previous_chunk_num = chunk_index - 1 if chunk_index > 0 else None
        next_chunk_num = (
            chunk_index + 1 if chunk_index < total_chunks - 1 else None
        )                                               # F

        expanded_chunk_parts = []

        if previous_chunk_num is not None:
            expanded_chunk_parts.append(
                granular_chunks[previous_chunk_num].page_content
            )

        expanded_chunk_parts.append(granular_chunk.page_content)

        if next_chunk_num is not None:
            expanded_chunk_parts.append(
                granular_chunks[next_chunk_num].page_content
            )

        expanded_chunk_text = "\n\n".join(expanded_chunk_parts)  # G

        expanded_chunk_id = str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"{destination_url}#expanded={chunk_index}",
            )
        )                                               # H

        granular_chunk_id = str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"{destination_url}#granular={chunk_index}",
            )
        )
        granular_chunk_ids.append(granular_chunk_id)

        expanded_metadata = {
            **granular_chunk.metadata,
            "source": destination_url,
            "central_chunk_index": chunk_index,
        }
        expanded_metadata.pop(doc_key, None)

        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text,
            metadata=expanded_metadata,
        )                                               # I

        expanded_chunk_store_items.append(
            (expanded_chunk_id, expanded_chunk_doc)
        )

        granular_chunk.metadata.update(
            {
                doc_key: expanded_chunk_id,
                "source": destination_url,
                "central_chunk_index": chunk_index,
            }
        )                                               # J

    print(
        f"Ingesting {destination_url}: "
        f"{total_chunks} granular and expanded chunks"
    )

    add_granular_documents_in_batches(
        vectorstore=multi_vector_retriever.vectorstore,
        documents=granular_chunks,
        document_ids=granular_chunk_ids,
        batch_size=8,
    )                                                   # K

    # Store expanded chunks only after all granular vectors succeed.
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items
    )                                                   # L

    print(f"Finished ingesting {destination_url}")

# A Load one destination page.
# B Retrieve its HTML document.
# C Transform HTML into clean text.
# D Split the destination into granular chunks.
# E Iterate over the granular chunks.
# F Identify the previous and next chunk without crossing page boundaries.
# G Expand the current chunk with its available neighbors.
# H Assign a deterministic ID to the expanded chunk.
# I Preserve source metadata on the expanded chunk.
# J Link the granular vector to its expanded chunk.
# K Upsert granular chunks in small retryable batches.
# L Store expanded chunks after their granular vectors succeed.


Loading https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.18it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall: 125 granular and expanded chunks
Embedded granular chunks 1-8/125
Embedded granular chunks 9-16/125
Embedded granular chunks 17-24/125
Embedded granular chunks 25-32/125
Embedded granular chunks 33-40/125
Embedded granular chunks 41-48/125
Embedded granular chunks 49-56/125
Embedded granular chunks 57-64/125
Embedded granular chunks 65-72/125
Embedded granular chunks 73-80/125
Embedded granular chunks 81-88/125
Embedded granular chunks 89-96/125
Embedded granular chunks 97-104/125
Embedded granular chunks 105-112/125
Embedded granular chunks 113-120/125
Embedded granular chunks 121-125/125
Finished ingesting https://en.wikivoyage.org/wiki/Cornwall

Loading https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.96it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall: 43 granular and expanded chunks
Embedded granular chunks 1-8/43
Embedded granular chunks 9-16/43
Embedded granular chunks 17-24/43
Embedded granular chunks 25-32/43
Embedded granular chunks 33-40/43
Embedded granular chunks 41-43/43
Finished ingesting https://en.wikivoyage.org/wiki/North_Cornwall

Loading https://en.wikivoyage.org/wiki/South_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.00it/s]


Ingesting https://en.wikivoyage.org/wiki/South_Cornwall: 31 granular and expanded chunks
Embedded granular chunks 1-8/31
Embedded granular chunks 9-16/31
Embedded granular chunks 17-24/31
Embedded granular chunks 25-31/31
Finished ingesting https://en.wikivoyage.org/wiki/South_Cornwall

Loading https://en.wikivoyage.org/wiki/West_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.06it/s]


Ingesting https://en.wikivoyage.org/wiki/West_Cornwall: 39 granular and expanded chunks
Embedded granular chunks 1-8/39
Embedded granular chunks 9-16/39
Embedded granular chunks 17-24/39
Embedded granular chunks 25-32/39
Embedded granular chunks 33-39/39
Finished ingesting https://en.wikivoyage.org/wiki/West_Cornwall


In [5]:
# Performing a search on granular information

retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")

print(len(retrieved_docs))
print(retrieved_docs[0])

4
page_content='Buses only serve designated stops when in towns; otherwise, you can flag them
down anywhere that's safe for them to stop.

### By train

[edit]

**CrossCountry Trains** and **Great Western Railway** operate regular train
services between the main centres of population, the latter company also
serving a number of other towns on branch lines. For train times and fares
visit National Rail Enquiries.

The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and
Plymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for
under-16s.

There is also a new Pay as you Go smartcard system for Cornwall, run by GWR
but valid on all trains in Cornwall.

### By ferry/boat

[edit]

There is also a new Pay as you Go smartcard system for Cornwall, run by GWR
but valid on all trains in Cornwall.

### By ferry/boat

[edit]

In certain areas of Cornwall, ferries exist. They can be considered largely
separate from other public transport as they are run by pr

In [6]:
# Comparing with direct semantic search on granular chunks

granular_docs_only = granular_chunks_collection.similarity_search(
    "Cornwall Ranger"
)

print(len(granular_docs_only))
print(granular_docs_only[0])

4
page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and
Plymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for
under-16s.

There is also a new Pay as you Go smartcard system for Cornwall, run by GWR
but valid on all trains in Cornwall.

### By ferry/boat

[edit]' metadata={'doc_id': 'd920071c-37e3-510c-8431-e105fad3e689', 'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'language': 'en', 'central_chunk_index': 66, 'title': 'Cornwall – Travel guide at Wikivoyage'}


The section ran successfully and the `MultiVectorRetriever` mapping works correctly.

Ingestion evidence:

| Destination | Granular/expanded chunks | Embedding batches |
|---|---:|---:|
| Cornwall | 125 | 16 |
| North Cornwall | 43 | 6 |
| South Cornwall | 31 | 4 |
| West Cornwall | 39 | 5 |
| **Total** | **238** | **31** |

No retries or errors appeared, so every BGE-M3 embedding batch succeeded initially.

For `"Cornwall Ranger"`:

- Direct granular search selected chunk index `66`.
- Its metadata contains the expanded-document link:

```text
doc_id: d920071c-37e3-510c-8431-e105fad3e689
central_chunk_index: 66
```

- `MultiVectorRetriever` followed that ID and returned the expanded document for central chunk `66`.
- The direct result contains only the focused Cornwall Ranger passage.
- The expanded result additionally includes the preceding train context and following ferry context.

This conclusively verifies:

```text
query
→ matching granular vector
→ expanded-document doc_id
→ previous + current + next context
```

One quality issue is visible: the expanded result repeats the smartcard and ferry transition. This is because `RecursiveCharacterTextSplitter` defaults to `chunk_overlap=200`, so neighboring chunks already share text before they are combined. This is a reasonable baseline, but it may require *fine-tuning* to find the optimal balance between context continuity, duplication and retrieval quality for the specific data and use case.

